In [7]:
# -*- coding: utf-8 -*-
"""
Detecção Automatizada de Embarcações de Garimpo Ilegal - INFERÊNCIA RÁPIDA
Autor: Fellipe Machado Castro (Universidade Federal do Pará - UFPA)
"""
print("Instalando bibliotecas necessárias...")
!pip install -q rasterio geopandas segmentation-models-pytorch

import os, glob
import numpy as np
import rasterio
import geopandas as gpd
from rasterio.features import rasterize
from rasterio.windows import Window
from shapely.geometry import box
import torch
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from scipy.ndimage import label, find_objects, binary_opening
from scipy.signal.windows import gaussian as gaussian_window
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

Instalando bibliotecas necessárias...


In [10]:
BASE_DIR = "/content/drive/MyDrive/Dados_TCC_Garimpo"
RESULT_DIR = os.path.join(BASE_DIR, "resultados_inferencia_oficial")
os.makedirs(RESULT_DIR, exist_ok=True)

ENCODER_NAME = "resnet34"  # resnet34 (padrão) ou resnet50
MODEL_SAVE_PATH = os.path.join(BASE_DIR, f"unet_{ENCODER_NAME}.pth")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Aplica automaticamente o limiar otimizado de cada rede no TCC
if ENCODER_NAME == "resnet34":
    BEST_THR = 0.40
elif ENCODER_NAME == "resnet50":
    BEST_THR = 0.45

CHIP_SIZE = 256
STRIDE = 128

print(f"Dispositivo: {DEVICE.upper()}")
print("Carregando arquitetura e pesos do modelo treinado...")

model = smp.Unet(
    encoder_name=ENCODER_NAME,
    encoder_weights=None,
    in_channels=1,
    classes=1
).to(DEVICE)

model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=DEVICE))
model.eval()

print("Modelo pronto para uso!")

Dispositivo: CUDA
Carregando arquitetura e pesos do modelo treinado...
Modelo pronto para uso!


In [11]:
def normalize_sar(img):
    img = img.astype(np.float32)
    img = np.where(img < -999, np.nan, img)
    img = np.nan_to_num(img, nan=np.nan)
    finite = img[np.isfinite(img)]
    if finite.size == 0: return np.zeros_like(img, dtype=np.float32)
    lo, hi = np.percentile(finite, 2), np.percentile(finite, 99)
    img = np.nan_to_num(img, nan=lo)
    img = np.clip((img - lo) / (hi - lo + 1e-6), 0, 1)
    return img.astype(np.float32)

def predict_tta(model, x):
    outs = []
    for k in range(4):
        xx = torch.rot90(x, k, dims=[2,3])
        with torch.no_grad(): p = torch.sigmoid(model(xx))
        outs.append(torch.rot90(p, -k, dims=[2,3]))
    with torch.no_grad(): p = torch.sigmoid(model(torch.flip(x, dims=[3])))
    outs.append(torch.flip(p, dims=[3]))
    return torch.stack(outs).mean(0)

TIF_FILES = glob.glob(os.path.join(BASE_DIR, "imagens_sar_originais", "S1_*_2025_8.tif"))
_g1d = gaussian_window(CHIP_SIZE, std=CHIP_SIZE/4)
GAUSS_KERNEL = np.outer(_g1d, _g1d).astype(np.float32)
GAUSS_KERNEL /= GAUSS_KERNEL.max()

print(f"Reconstruindo e avaliando {len(TIF_FILES)} imagens de validação (Agosto/2025)...\n")

total_TP = total_FP = total_FN = total_GT = 0

for tif_file in TIF_FILES:
    fname = os.path.basename(tif_file)
    try:
        parts = fname.replace(".tif","").split("_"); city, year, month = parts[1], parts[2], parts[3]
    except: continue

    gj_path = os.path.join(BASE_DIR, f"gabarito_{year}_{month.zfill(2)}.geojson")
    if not os.path.exists(gj_path):
        gj_path = os.path.join(BASE_DIR, f"gabarito_{year}_{month}.geojson")
        if not os.path.exists(gj_path): continue

    with rasterio.open(tif_file) as src:
        H, W = src.height, src.width
        full_img = src.read(1)

        gdf = gpd.read_file(gj_path)
        if src.crs: gdf = gdf.to_crs(src.crs)
        bbox = box(*src.bounds)
        gdf = gdf[gdf.geometry.intersects(bbox)]

        if not gdf.empty:
            gdf['geometry'] = gdf.geometry.buffer(30 / 111320) if src.crs.is_geographic else gdf.geometry.buffer(30)
            shapes = ((geom, 1) for geom in gdf.geometry)
            mask_gt = rasterize(shapes, out_shape=(H, W), transform=src.transform, fill=0, dtype=np.uint8)
        else:
            mask_gt = np.zeros((H, W), dtype=np.uint8)

        prob_map   = np.zeros((H,W), dtype=np.float32)
        weight_map = np.zeros((H,W), dtype=np.float32)

        for y in range(0, H, STRIDE):
            for x in range(0, W, STRIDE):
                window = Window(x, y, CHIP_SIZE, CHIP_SIZE).intersection(Window(0,0,W,H))
                chip = src.read(1, window=window)
                if chip.size == 0: continue
                hh, ww = chip.shape
                pad = np.zeros((CHIP_SIZE, CHIP_SIZE), dtype=np.float32)
                pad[:hh,:ww] = chip
                norm = normalize_sar(pad)
                pred = predict_tta(model, torch.from_numpy(norm).unsqueeze(0).unsqueeze(0).to(DEVICE)).cpu().numpy()[0,0]
                ker  = GAUSS_KERNEL[:hh,:ww]
                prob_map[y:y+hh, x:x+ww]   += pred[:hh,:ww]*ker
                weight_map[y:y+hh, x:x+ww] += ker

        prob_map /= (weight_map + 1e-6)
        binary_mask = binary_opening(prob_map > BEST_THR, iterations=1)

        lbl_gt, n_gt = label(mask_gt)
        lbl_pr, n_pr = label(binary_mask)

        objs_gt = [set(zip(*np.where(lbl_gt == i))) for i in range(1, n_gt+1) if np.sum(lbl_gt == i) >= 8]
        objs_pr = [set(zip(*np.where(lbl_pr == i))) for i in range(1, n_pr+1) if np.sum(lbl_pr == i) >= 8]

        TP = FP = FN = 0
        pred_px = set().union(*objs_pr) if objs_pr else set()
        gt_px = set().union(*objs_gt) if objs_gt else set()

        for gset in objs_gt:
            if gset & pred_px: TP += 1
            else: FN += 1
        for pset in objs_pr:
            if not (pset & gt_px): FP += 1

        total_TP += TP; total_FP += FP; total_FN += FN; total_GT += len(objs_gt)

        print(f" -> {city}: Gabarito={len(objs_gt)} | Acertou(TP)={TP} | Falso Alarme(FP)={FP} | Perdeu(FN)={FN}")

        plt.figure(figsize=(15,15))
        plt.imshow(normalize_sar(full_img), cmap='gray'); ax = plt.gca()
        valid = [o for o in find_objects(lbl_pr) if (o[0].stop-o[0].start)*(o[1].stop-o[1].start) >= 12]
        for o in valid:
            y0,y1 = o[0].start, o[0].stop; x0,x1 = o[1].start, o[1].stop
            ax.add_patch(patches.Rectangle((x0,y0), x1-x0, y1-y0, linewidth=2, edgecolor='red', facecolor='none'))
        plt.title(f"{len(valid)} balsas | Detecções na Cena Inteira | {fname} (Limiar={BEST_THR:.2f})", fontsize=14)
        plt.axis('off')
        plt.savefig(os.path.join(RESULT_DIR, f"INFERENCIA_{fname}.png"), bbox_inches='tight', dpi=150); plt.close()

precisao = total_TP / (total_TP + total_FP + 1e-6)
recall = total_TP / (total_TP + total_FN + 1e-6)
f1_score = 2 * precisao * recall / (precisao + recall + 1e-6)

print("\n" + "="*50)
print("MÉTRICAS FINAIS - CENA COMPLETA")
print("="*50)
print(f"Total de Balsas Reais (Gabarito): {total_GT}")
print(f"Verdadeiros Positivos (TP)    : {total_TP}")
print(f"Falsos Positivos (FP)         : {total_FP}")
print(f"Falsos Negativos (FN)         : {total_FN}")
print("-" * 50)
print(f"Precisão Matemática : {precisao:.3f}")
print(f"Recall Matemático   : {recall:.3f}")
print(f"F1-Score Matemático : {f1_score:.3f}")
print("="*50)
print("Varredura concluída! Imagens geradas na pasta resultados_inferencia.")

Reconstruindo e avaliando 5 imagens de validação (Agosto/2025)...

 -> SaoCarlos: Gabarito=0 | Acertou(TP)=0 | Falso Alarme(FP)=0 | Perdeu(FN)=0
 -> PortoVelho: Gabarito=15 | Acertou(TP)=13 | Falso Alarme(FP)=5 | Perdeu(FN)=2
 -> JaciParana: Gabarito=14 | Acertou(TP)=14 | Falso Alarme(FP)=1 | Perdeu(FN)=0
 -> ControleHumaita: Gabarito=0 | Acertou(TP)=0 | Falso Alarme(FP)=0 | Perdeu(FN)=0
 -> BoaHora: Gabarito=2 | Acertou(TP)=2 | Falso Alarme(FP)=3 | Perdeu(FN)=0

MÉTRICAS FINAIS - CENA COMPLETA
Total de Balsas Reais (Gabarito): 31
Verdadeiros Positivos (TP)    : 29
Falsos Positivos (FP)         : 9
Falsos Negativos (FN)         : 2
--------------------------------------------------
Precisão Matemática : 0.763
Recall Matemático   : 0.935
F1-Score Matemático : 0.841
Varredura concluída! Imagens geradas na pasta resultados_inferencia.
